# Customer Churn Analysis
**Dataset:** IBM Telco Customer Churn (Kaggle)  
**Goal:** Identify customers likely to churn and understand the key drivers.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette(['#3498db', '#e74c3c'])

print('Libraries loaded ✓')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_rate   = churn_counts['Yes'] / len(df) * 100
print(f'Churn rate: {churn_rate:.1f}%')
print(churn_counts)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pie
axes[0].pie(churn_counts, labels=['Retained', 'Churned'],
            colors=['#3498db', '#e74c3c'], autopct='%1.1f%%', startangle=140)
axes[0].set_title('Overall Churn Split', fontweight='bold')

# Contract type
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'],
                    edgecolor='none', width=0.6)
axes[1].set_title('Churn by Contract Type', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right')
axes[1].legend(['Retained', 'Churned'])

# Tenure
for label, color in [('No', '#3498db'), ('Yes', '#e74c3c')]:
    axes[2].hist(df[df['Churn'] == label]['tenure'],
                 bins=24, alpha=0.6, color=color,
                 label='Retained' if label == 'No' else 'Churned',
                 edgecolor='none')
axes[2].set_title('Churn by Tenure', fontweight='bold')
axes[2].set_xlabel('Tenure (months)')
axes[2].legend()

plt.tight_layout()
plt.savefig('../assets/eda_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly charges distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color in [('No', '#3498db'), ('Yes', '#e74c3c')]:
    df[df['Churn'] == label]['MonthlyCharges'].plot.kde(
        ax=axes[0], color=color, linewidth=2,
        label='Retained' if label == 'No' else 'Churned'
    )
axes[0].set_title('Monthly Charges by Churn Status', fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].legend()

# Internet service
internet_churn = df.groupby(['InternetService', 'Churn']).size().unstack(fill_value=0)
internet_churn_pct = internet_churn.div(internet_churn.sum(axis=1), axis=0) * 100
internet_churn_pct.plot(kind='barh', ax=axes[1],
                         color=['#3498db', '#e74c3c'], edgecolor='none', width=0.55)
axes[1].set_title('Churn Rate by Internet Service (%)', fontweight='bold')
axes[1].legend(['Retained', 'Churned'])

plt.tight_layout()
plt.savefig('../assets/eda_charges.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
import sys; sys.path.insert(0, '../src')
from churn_model import preprocess

df_clean = preprocess(df.copy())

plt.figure(figsize=(12, 8))
corr = df_clean.corr()[['Churn']].sort_values('Churn', ascending=False)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, linewidths=.5, cbar_kws={'shrink': 0.5})
plt.title('Feature Correlation with Churn', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../assets/correlation.png', bbox_inches='tight')
plt.show()

## 3. Model Training

In [ ]:
from churn_model import train, evaluate, feature_importance, save_model

model, X_train, X_test, y_train, y_test = train(df_clean)
metrics, cm, report = evaluate(model, X_test, y_test)

print('Model metrics:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v}%')

## 4. Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Retained', 'Churned'])
disp.plot(ax=axes[0], cmap='RdYlGn', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC Curve
y_prob = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#e74c3c')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('../assets/model_eval.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature importances
fi = feature_importance(model, X_train).head(15).sort_values('importance')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v > fi['importance'].quantile(0.7) else '#3498db'
          for v in fi['importance']]
ax.barh(fi['feature'], fi['importance'], color=colors, edgecolor='none')
ax.set_title('Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig('../assets/feature_importance.png', bbox_inches='tight')
plt.show()

## 5. Save Model

In [ ]:
save_model(model)
print('Done! Model saved to ../model/churn_model.pkl')